# Talk to your LLM

Use this notebook with [the practical](PRACTICAL.md). You will send chat requests to your Nebius endpoint, compare its base model and LoRA adapter, then try a larger hosted model through Nebius Token Factory.


## The OpenAI Python SDK

The **SDK** is a Python library that builds and sends the API request for you. You give it a server address, an API key, a model name, and a list of messages. The server returns a response object; the generated text is in `choices[0].message.content`.

Both vLLM and Token Factory offer an **OpenAI-compatible API**, so the shape of the chat request is almost the same. The endpoint URL and authentication key change when you switch services.


In [ ]:
%pip install -q "openai>=1.30,<3"


## Connect to your Nebius endpoint

Copy the HTTPS URL from **Network → Public endpoints** in the Nebius console. Authentication is disabled for the class endpoint, so the SDK needs only a placeholder key. Run this cell again with the adapted endpoint URL after training.


In [ ]:
from openai import OpenAI

endpoint_url = input("Endpoint URL: ").strip().rstrip("/")
endpoint_url = endpoint_url.removesuffix("/v1")
client = OpenAI(
    base_url=f"{endpoint_url}/v1",
    api_key="not-needed",
)
served_models = [model.id for model in client.models.list().data]
base_model = next(model for model in served_models if model != "character")
print("Models served by this endpoint:", served_models)


## Send a chat request

A chat request is a list of messages. Each message has a `role` and `content`: `user` is what you ask, and `assistant` is what the model returns. A completion call sends those messages to the selected model.


In [ ]:
messages = [
    {"role": "user", "content": "How do I make a pizza?"},
]

reply = client.chat.completions.create(
    model=base_model,
    messages=messages,
    max_tokens=160,
    temperature=0.3,
)
print(reply.choices[0].message.content)


Try a few different questions by changing `content`. For example, ask who the model is, how it would solve a mystery, or what it knows about LoRA. The request code stays the same; only the messages or model name need to change.


In [ ]:
my_question = "Can you explain LoRA in simple terms?"
reply = client.chat.completions.create(
    model=base_model,
    messages=[{"role": "user", "content": my_question}],
    max_tokens=180,
    temperature=0.3,
)
print(reply.choices[0].message.content)


## Compare the base model and adapter

After training, connect to the **new** endpoint by rerunning the connection cell above. Its model list should include the base model and `character`. Send the same messages to each so you can see how their answers differ.


In [ ]:
question = "I have to make a difficult choice. What do you suggest?"
messages = [{"role": "user", "content": question}]

for model_name in [base_model, "character"]:
    reply = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=180,
        temperature=0.3,
    )
    print(f"\n--- {model_name} ---\n{reply.choices[0].message.content}")


## Try a larger model with Token Factory

Token Factory hosts models behind another OpenAI-compatible API. They do not use the L40S endpoint you created. Instead, you send a request to Token Factory's API address and authenticate with the class API key. You can see the model IDs available to your key below; choose a large model from that list.

The key is requested with hidden input. It is used in this notebook's memory and is not included in the code cell or its output. Do not paste it into a cell or save it in the notebook.


In [ ]:
from getpass import getpass

token_factory = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=getpass("Token Factory API key: "),
)
tofa_models = token_factory.models.list().data
print("Models available to this key:")
for model in tofa_models:
    print("-", model.id)


For Token Factory, add a `system` message before the `user` message. The system message sets the requested voice; it is a prompt, not a trained adapter. Try changing the first message to match your chosen character. For Shakespeare, try: `Speak as a character in a Shakespeare play, using Early Modern English.`

Use the same question you asked the adapter. Does this larger model produce a similar voice from the system prompt alone? A familiar character or style may appear in the model's pretraining, but the prompt does not guarantee accurate knowledge or a perfect imitation.


In [ ]:
# Choose a large model ID printed above.
tofa_model = input("Token Factory model ID: ").strip()

system_prompt = (
    "You are Sherlock Holmes. Speak with measured, precise observations, "
    "notice small details, and explain your reasoning in a restrained Victorian voice."
)
# For Shakespeare, replace the prompt above with:
# system_prompt = "Speak as a character in a Shakespeare play, using Early Modern English."

reply = token_factory.chat.completions.create(
    model=tofa_model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ],
    max_tokens=220,
    temperature=0.7,
)
print(reply.choices[0].message.content)
